# Exploratory Data Analysis of FiQA-2018


In [18]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 200)


DATA_DIR = Path("../data/0.raw_fiqa")
INDEX_DIR = Path("../data/indexes/bm25")

## Corpus

The fixed set of documents the pipeline searches over. FiQA docs are forum posts, which have no title — but BEIR ships every corpus with the same `_id` / `title` / `text` schema (datasets like SciFact fill `title` with the paper title), so here `title` is empty on every row and retrieval uses `text` alone.

In [19]:
corpus = pd.read_parquet(DATA_DIR / "corpus.parquet")
print(f"{len(corpus):,} documents.")
print(f"Columns: {list(corpus.columns)}.")
corpus.head(3)

57,638 documents.
Columns: ['_id', 'title', 'text'].


,_id,title,text
0,3,,"I'm not saying I don't like the idea of on-the-job training too, but you can't expect the company to do that. Training workers is not their job - they're building software. Perhaps educational sys..."
1,31,,"So nothing preventing false ratings besides additional scrutiny from the market/investors, but there are some newer controls in place to prevent institutions from using them. Under the DFA banks c..."
2,56,,"You can never use a health FSA for individual health insurance premiums. Moreover, FSA plan sponsors can limit what they are will to reimburse. While you can't use a health FSA for premiums, you..."


In [20]:
# A document, in full.
corpus["text"].iloc[0]

"I'm not saying I don't like the idea of on-the-job training too, but you can't expect the company to do that. Training workers is not their job - they're building software. Perhaps educational systems in the U.S. (or their students) should worry a little about getting marketable skills in exchange for their massive investment in education, rather than getting out with thousands in student debt and then complaining that they aren't qualified to do anything."

In [21]:
# Check if all titles are empty.
all_titles_empty = (corpus["title"] == "").all()
print(all_titles_empty)

True


## Queries

Natural-language finance questions posed against the corpus. Same BEIR schema as the corpus, so `title` is again empty — the question text lives in `text`.

In [22]:
queries = pd.read_parquet(DATA_DIR / "queries.parquet")
print(f"{len(queries):,} queries. Columns: {list(queries.columns)}")
print(queries["text"].iloc[0])
queries.head(3)

6,648 queries. Columns: ['_id', 'title', 'text']
What is considered a business expense on a business trip?


,_id,title,text
0,0,,What is considered a business expense on a business trip?
1,4,,Business Expense - Car Insurance Deductible For Accident That Occurred During a Business Trip
2,5,,Starting a new online business


## Qrels (ground truth)

Each qrel is a `(query-id, corpus-id, score)` triple asserting that the document answers the query. This is the only source of truth for measuring retrieval quality.

FiQA uses **binary relevance**: `score` is always `1`. Pairs absent from the table are implicitly `0` (not relevant) — irrelevant pairs are never stored. Other BEIR datasets (e.g. TREC-COVID) use graded scores like `0/1/2`.

In [23]:
qrels = pd.read_parquet(DATA_DIR / "qrels.parquet")
print(f"{len(qrels):,} relevance judgments. Columns: {list(qrels.columns)}")
qrels.head(3)

1,706 relevance judgments. Columns: ['query-id', 'corpus-id', 'score']


,query-id,corpus-id,score
0,8,566392,1
1,8,65404,1
2,15,325273,1


In [31]:
print(f"Value counts for qrels: {qrels['score'].value_counts()}")
print(f"Unique values in qrels['score']: {qrels['score'].unique()}")
print(f"Min and max values in qrels['score']: {qrels['score'].min()}, {qrels['score'].max()}")


Value counts for qrels: score
1    1706
Name: count, dtype: int64
Unique values in qrels['score']: [1]
Min and max values in qrels['score']: 1, 1


## Test queries

A query is *evaluable* only if it has at least one qrel. That subset — the **test queries** — is the only thing the pipeline can score.

In [25]:
test_query_ids = set(qrels["query-id"].astype(str))
test_queries = queries[queries["_id"].astype(str).isin(test_query_ids)]
print(f"Test queries (with qrels): {len(test_queries):,} of {len(queries):,} total")

Test queries (with qrels): 648 of 6,648 total


## One query and its relevant documents

Inspect a single test query and the corpus documents its qrels mark as relevant.

In [34]:
query = test_queries.iloc[0]
print("Example test query")
print(f"  Test query _id:  {query['_id']}")
print(f"  text: {query['text']}")

relevant = qrels[qrels["query-id"].astype(str) == str(query["_id"])]
print(f"\n  Relevant docs (qrels) for this question: {list(relevant['corpus-id'])}")

Example test query
  Test query _id:  4641
  text: Where should I park my rainy-day / emergency fund?

  Relevant docs (qrels) for this question: [44594, 406219, 319954, 397358, 88327]
